In [33]:
'''
Steps:

- load the data from dataset_full.csv
- create a dataframe compressing data through PCA, df_full_pca
- split x and three Ys for df_full_pca
- create a dataframe compressing data through LDA, df_full_lda
- split x and three Ys for df_full_lda
- first test cycle with cross-validation: PCA + LR, RF, MLP (without and with Label Powerset) 
- first test cycle with cross-validation: LDA + LR, RF, MLP (without and with Label Powerset) 
- second test cycle with cross-validation: LDA + SVM, SGD, KNN, NB, DT
- third test cycle with cross-validation: LDA + VotingClassifier(3) with (LR, RF, MLP, SVM, SGD, KNN, NB, DT)
- third test cycle with cross-validation: LDA + GradientBoostingClassifier

The variance for PCA has been set to 0.90, but it can be modified with the variable comps. 
The number of folds for cross-validating is 5, but it can be modified with the variable no_folds
Seed = 123

'''

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from numpy import set_printoptions
import time
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from skmultilearn.problem_transform import LabelPowerset
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
from itertools import combinations
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import svm

formatter = "{0:.3f}"

pd.options.display.float_format = '{:,.3f}'.format 

# uncomment to show every output rows
pd.set_option('display.max_rows', 1000)

# uncomment to show every output columns
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# how the floating numbers are shown in the numpy arrays
set_printoptions(precision=3) 

# how the floating numbers are shown in pandas
pd.options.display.float_format = '{:,.3f}'.format  

np.random.seed(123)


In [2]:
''' Functions '''

def run_crossvalidation_with_multilabel(no_folds, model, x, Y_labels):
    '''
    It runs a k-fold cross validation using a multilabel model and computes the weighted 
    accuracy, precision, recall, f1 for all the classes in Y
    :param no_folds: number of folds
    :model the algorithm chosen for training initialised with Label Powerset
    :x the features
    :Y the targets
    :return: weighted accuracy, precision, recall, f1
    
    '''
    cross_val = KFold(n_splits=no_folds, shuffle=True, random_state=123)
    
    accuracy = 0.0
    precision = 0.0
    recall = 0.0
    f1 = 0.0
    
    for train, test in cross_val.split(x):
        # split train and test data
        train_index = list(train)
        test_index = list(test)

        x_train, Y_train = x.iloc[train_index, :], Y_labels.iloc[train_index, :]
        x_test, Y_test = x.iloc[test_index, :], Y_labels.iloc[test_index, :]

        # train the model
        model.fit(x_train, Y_train)
    
        # predict on test
        predict_on_test = model.predict(x_test)
        # The following 2 lines can be used to print the confusion matrix and the classification report
        #print(confusion_matrix(Y_train, predict_on_test))
        #print(classification_report(Y_test, predict_on_test))
        acc_result = accuracy_score(Y_test, predict_on_test)
        prec_result = precision_score(Y_test, predict_on_test, average='weighted') # or samples
        rec_result = recall_score(Y_test, predict_on_test, average='weighted')
        f1_result = f1_score(Y_test, predict_on_test, average='weighted')

        accuracy = accuracy + acc_result
        precision = precision + prec_result
        recall = recall + rec_result
        f1 = f1 + f1_result  
        
        return formatter.format(accuracy), formatter.format(precision), formatter.format(recall), formatter.format(f1) 
    
def run_cross_validation(no_folds, model, x, Y):
    '''
    It runs a k-fold cross validation using a model and compute the average
    accuracy, precision, recall, f1. 
    :param no_folds: number of folds
    :model the algorithm chosen for training 
    :x the features
    :Y the target
    :return: averaged accuracy, precision, recall, f1
    
    '''
    cross_val = KFold(n_splits=no_folds, shuffle=True, random_state=123)
    
    accuracy = 0.0
    precision = 0.0
    recall = 0.0
    f1 = 0.0
    
    for train, test in cross_val.split(x):
        # split train and test data
        train_index = list(train)
        test_index = list(test)

        x_train, Y_train = x.iloc[train_index, :], Y.iloc[train_index, :]
        x_test, Y_test = x.iloc[test_index, :], Y.iloc[test_index, :]

        # train the model
        model.fit(x_train, Y_train)
    
        # predict on test
        predict_on_test = model.predict(x_test)
        # The following 2 lines can be used to print the confusion matrix and the classification report
        #print(confusion_matrix(Y_train, predict_on_test))
        #print(classification_report(Y_test, predict_on_test))
        acc_result = accuracy_score(Y_test, predict_on_test)
        prec_result = precision_score(Y_test, predict_on_test)
        rec_result = recall_score(Y_test, predict_on_test)
        f1_result = f1_score(Y_test, predict_on_test)

        accuracy = accuracy + acc_result
        precision = precision + prec_result
        recall = recall + rec_result
        f1 = f1 + f1_result
    
    # average the scores; precision, recall, f1 are computed on the basis of the label 1 
    accuracy = accuracy / no_folds
    precision = precision / no_folds
    recall = recall / no_folds
    f1 = f1 / no_folds
    
    return formatter.format(accuracy), formatter.format(precision), formatter.format(recall), formatter.format(f1)    

def create_multiindex_dataframe(filename, index_list):
    '''
    Create a multiindex dataframe from a csv or xlsx file
    :param filename: a csv or xlsx file
    :param index_list: number of column levels, usually [0, 1]
    :return: a multiindex dataframe
    '''

    if "xlsx" in filename:
        df = pd.read_excel(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)
    elif "csv" in filename:
        df = pd.read_csv(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)

    return df

def run_pca_with_three_targets(df, comps):
    '''
    Run the PCA with a multiindex dataframe having three targets. 
    Average explained variance and components, then create a df
    
    :param df: a pandas dataframe
    :param comps: the number of components for PCA or the % of variance explained
    :return: the explained variance and a pandas dataframe with averaged PCA components
    
    '''

    # split features and targets, extract column names and indexes
    col_names = df.columns.values
    x1 = df[col_names[0:-3]]
    Y1 = df[col_names[-3]].rename("Class 1")
    ind = x1.index.values
    
    x2 = df[col_names[0:-2]]
    Y2 = df[col_names[-2]].rename("Class 2")

    x3 = df[col_names[0:-1]]
    Y3 = df[col_names[-1]].rename("Class 3")
    
    # Fit and transform
    pca1 = PCA(n_components=comps, svd_solver='full')
    principal_components1 = pca1.fit_transform(x1)
    explained_var1 = pca1.explained_variance_ratio_
    
    pca2 = PCA(n_components=comps, svd_solver='full')
    principal_components2 = pca2.fit_transform(x2)
    explained_var2 = pca2.explained_variance_ratio_
    
    pca3 = PCA(n_components=comps, svd_solver='full')
    principal_components3 = pca3.fit_transform(x3)
    explained_var3 = pca3.explained_variance_ratio_
    
    # Sum and average the explained variance
    explained_var = ( explained_var1 + explained_var2 + explained_var3 ) / 3

    # create a list with the component names
    cols = []
    #for i in range(1, comps + 1):
    for i in range(1, pca1.n_components_ + 1):
        cols.append('Eigenvector %d' % i)

    # Average the components and create a new dataframe with the them
    principal_components = ( principal_components1 + principal_components2 + principal_components3 ) / 3
    pca_df = pd.DataFrame(data=principal_components, index=ind, columns=cols)

    # Concat the pca_df with Y
    final_df = pd.concat([pca_df, Y1], axis=1)
    final_df = pd.concat([final_df, Y2], axis=1)
    final_df = pd.concat([final_df, Y3], axis=1)

    return explained_var, final_df

def run_lda_with_three_targets(df):
    '''
    Run the LDA with a multiindex dataframe having three targets. 
    Create a df with a discriminant for each target and the 3 Ys
    
    :param df: a pandas dataframe
    :param comps: the number of components for PCA or the % of variance explained
    :return: the explained variance and a pandas dataframe with the PCA components    
    
    '''
    # split features and targets, extract column names and indexes
    col_names = df.columns.values

    x1 = df[col_names[0:-3]]
    Y1 = df[col_names[-3]].rename("Class 1")

    # this will be used for the 3 LDAs
    ind = x1.index.values

    x2 = df[col_names[0:-2]]
    Y2 = df[col_names[-2]].rename("Class 2")
    
    x3 = df[col_names[0:-1]]
    Y3 = df[col_names[-1]].rename("Class 3")
    
    # fit and transform
    clf = LinearDiscriminantAnalysis()
    linear_discriminants1 = clf.fit_transform(x1, Y1)
    linear_discriminants2 = clf.fit_transform(x2, Y2)
    linear_discriminants3 = clf.fit_transform(x3, Y3)

    # create 3 separate df with the linear discriminants
    lda_df1 = pd.DataFrame(data=linear_discriminants1, index=ind, columns=['LDA1'])
    lda_df2 = pd.DataFrame(data=linear_discriminants2, index=ind, columns=['LDA2'])
    lda_df3 = pd.DataFrame(data=linear_discriminants3, index=ind, columns=['LDA3'])
    
    # concatenate the 3 dfs
    lda_final_df = pd.concat([lda_df1, lda_df2], axis=1)
    lda_final_df = pd.concat([lda_final_df, lda_df3], axis=1)

    # concatenate the 3 Ys
    lda_final_df = pd.concat([lda_final_df, Y1], axis=1)
    lda_final_df = pd.concat([lda_final_df, Y2], axis=1)
    lda_final_df = pd.concat([lda_final_df, Y3], axis=1)
    
    return lda_final_df


In [4]:
''' Dataframes and variables '''

file_full = 'dataset_full.csv'

index_list = [0, 1]

features =  ['adjusted_close', 'volume', 'totalRevenue', 'totalLiab', 'totalAssets', 'otherAssets', 'totalStockholderEquity', 
            'capitalExpenditures', 'incomeBeforeTax', 'researchDevelopment', 'incomeTaxExpense', 'netIncome', 
            'propertyPlantEquipment', 'netIncomeApplicableToCommonShares', 'sellingGeneralAdministrative', 'costOfRevenue', 
            'grossProfit', 'accountsPayable', 'operatingIncome', 'interestExpense', 'commonStock']

quarters = ["t0", "t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t10", "t11", "t12", "t13", "t14", "t15",
            "t16", "t17", "t18", "t19", "t20", "t21", "t22", "t23", "t24", "t25", "t26", "t27", "t28", "t29", "t30",
            "t31", "t32", "t33", "t34", "t35", "t36", "t37", "t38", "t39", "t40"]

comps = 0.90 # the amount of variance to keep for PCA
no_folds = 5 # the number of folds for cross-validating

# PCA feature extraction
# run the pca and store the variance and the new df
pca_results = run_pca_with_three_targets(create_multiindex_dataframe(file_full, index_list), comps)

pca_variance = pca_results[0]
df_full_pca = pca_results[1]

columns_pca = df_full_pca.columns.values # pca df column names

# separate x and Y for train_df
x_pca = df_full_pca.loc[:, columns_pca[0:18]] 
Y1_pca = df_full_pca.loc[:, "Class 1"].to_frame() # to_frame() convert data.series into a dataframe
Y2_pca = df_full_pca.loc[:, "Class 2"].to_frame()
Y3_pca = df_full_pca.loc[:, "Class 3"].to_frame()

# concatenate the 3 classes for PCA
Y_pca_all = pd.concat([Y1_pca, Y2_pca], axis=1)
Y_pca_all = pd.concat([Y_pca_all, Y3_pca], axis=1)

# LDA feature extraction
# run the lda and store the variance and the new df
df_full_lda = run_lda_with_three_targets(create_multiindex_dataframe(file_full, index_list))

columns_lda = df_full_lda.columns.values # lda df column names

# separate x and Y for train_df
x_lda = df_full_lda[columns_lda[0:3]] 
Y1_lda = df_full_lda.loc[:, "Class 1"].to_frame()
Y2_lda = df_full_lda.loc[:, "Class 2"].to_frame()
Y3_lda = df_full_lda.loc[:, "Class 3"].to_frame()

# concatenate the 3 classes for PCA
Y_lda_all = pd.concat([Y1_lda, Y2_lda], axis=1)
Y_lda_all = pd.concat([Y_lda_all, Y3_lda], axis=1)

targets_pca = [Y1_pca, Y2_pca, Y3_pca]
targets_lda = [Y1_lda, Y2_lda, Y3_lda]


In [15]:
''' FIRST TEST CYCLE '''
''' Cross-validation after PCA feature extraction: Logistic Regression '''

# several solvers have been tested: the best results with saga
logreg = LogisticRegression(solver='saga', random_state=123)

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_pca)):
    start = int(round(time.time() * 1000))
    avg_results_pca  = run_cross_validation(no_folds, logreg, x_pca, targets_pca[Y])
    print("PCA + LR: results for Y%s " %str(Y + 1) + " | " + avg_results_pca[0] + " | " + avg_results_pca[1] + " | " + avg_results_pca[2] + " | " + avg_results_pca[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))
    

                              Acc | Prec  |  Rec  |  F1
PCA + LR: results for Y1  | 0.506 | 0.536 | 0.538 | 0.477
PCA + LR: results for Y2  | 0.514 | 0.504 | 0.817 | 0.622
PCA + LR: results for Y3  | 0.502 | 0.529 | 0.641 | 0.507
ms 182


In [13]:
''' Cross-validation after PCA feature extraction: Logistic Regression and Label Powerset '''

# the approach used is Problem transformation with Label Powerset
# https://www.analyticsvidhya.com/blog/2017/08/introduction-to-multi-label-classification/

start = int(round(time.time() * 1000))
logreg_powerset = LabelPowerset(LogisticRegression(solver='saga', random_state=123))
print(run_crossvalidation_with_multilabel(no_folds, logreg_powerset, x_pca, Y_pca_all))
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.315', '0.523', '0.832', '0.642')
ms 169


In [18]:
''' Cross-validation after PCA feature extraction: Random Forests '''

randf = RandomForestClassifier(n_estimators=50, random_state=123)

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_pca)):
    start = int(round(time.time() * 1000))
    avg_results_pca  = run_cross_validation(no_folds, randf, x_pca, targets_pca[Y])
    print("PCA + RF: results for Y%s " %str(Y + 1) + " | " + avg_results_pca[0] + " | " + avg_results_pca[1] + " | " + avg_results_pca[2] + " | " + avg_results_pca[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))


                              Acc | Prec  |  Rec  |  F1
PCA + RF: results for Y1  | 0.537 | 0.517 | 0.448 | 0.476
PCA + RF: results for Y2  | 0.527 | 0.520 | 0.488 | 0.500
PCA + RF: results for Y3  | 0.534 | 0.511 | 0.437 | 0.467
ms 992


In [20]:
''' Cross-validation after PCA feature extraction: Random Forests and Label Powerset '''

start = int(round(time.time() * 1000))

randf_powerset = LabelPowerset(RandomForestClassifier(n_estimators=50, random_state=123))
print(run_crossvalidation_with_multilabel(no_folds, randf_powerset, x_pca, Y_pca_all))
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.293', '0.549', '0.457', '0.499')
ms 277


In [22]:
''' Cross-validation after PCA feature extraction: MLP Classifier '''

mlp = MLPClassifier(activation='relu', random_state=123, hidden_layer_sizes=(5, 5), max_iter=500)

print(" " * 30 + "Acc " + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_pca)):
    start = int(round(time.time() * 1000))
    avg_results_pca  = run_cross_validation(no_folds, mlp, x_pca, targets_pca[Y])
    print("PCA + MLP: results for Y%s " %str(Y + 1) + " | " + avg_results_pca[0] + " | " + avg_results_pca[1] + " | " + avg_results_pca[2] + " | " + avg_results_pca[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))


                              Acc  | Prec  |  Rec  |  F1
PCA + MLP: results for Y1  | 0.522 | 0.496 | 0.322 | 0.388
PCA + MLP: results for Y2  | 0.509 | 0.507 | 0.428 | 0.445
PCA + MLP: results for Y3  | 0.541 | 0.525 | 0.386 | 0.439
ms 6873


In [23]:
''' Cross-validation after PCA feature extraction: MLP Classifier and Label Powerset '''

start = int(round(time.time() * 1000))
mlp_powerset = LabelPowerset(MLPClassifier(activation='relu', random_state=123, hidden_layer_sizes=(5, 5), max_iter=500))
print(run_crossvalidation_with_multilabel(no_folds, mlp_powerset, x_pca, Y_pca_all))   
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.280', '0.482', '0.199', '0.282')
ms 1574


In [24]:
''' Cross-validation after LDA feature extraction: Logistic Regression ''' 
 
logreg = LogisticRegression(solver='saga', random_state=123)

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    avg_results_lda  = run_cross_validation(no_folds, logreg, x_lda, targets_lda[Y])
    print("LDA + LG: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))


                              Acc | Prec  |  Rec  |  F1
LDA + LG: results for Y1  | 0.919 | 0.918 | 0.909 | 0.913
LDA + LG: results for Y2  | 0.913 | 0.909 | 0.912 | 0.910
LDA + LG: results for Y3  | 0.912 | 0.905 | 0.911 | 0.907
ms 46


In [25]:
''' Cross-validation after LDA feature extraction: Logistic Regression and Label Powerset '''

start = int(round(time.time() * 1000))
logreg_powerset = LabelPowerset(LogisticRegression(solver='saga', random_state=123))
print(run_crossvalidation_with_multilabel(no_folds, logreg_powerset, x_lda, Y_lda_all))
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.804', '0.930', '0.914', '0.922')
ms 50


In [26]:
''' Cross-validation after LDA feature extraction: Random Forests ''' 

randf = RandomForestClassifier(n_estimators=50, random_state=123)

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    avg_results_lda  = run_cross_validation(no_folds, randf, x_lda, targets_lda[Y])
    print("LDA + RF: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))
    

                              Acc | Prec  |  Rec  |  F1
LDA + RF: results for Y1  | 0.942 | 0.939 | 0.938 | 0.939
LDA + RF: results for Y2  | 0.965 | 0.964 | 0.966 | 0.964
LDA + RF: results for Y3  | 0.913 | 0.908 | 0.909 | 0.908
ms 467


In [28]:
''' Cross-validation after LDA feature extraction: Random Forests and Label Powerset '''

start = int(round(time.time() * 1000))
randf_powerset = LabelPowerset(RandomForestClassifier(n_estimators=50, random_state=123))
print(run_crossvalidation_with_multilabel(no_folds, randf_powerset, x_lda, Y_lda_all))    
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.833', '0.945', '0.931', '0.938')
ms 132


In [30]:
''' Cross-validation after LDA feature extraction: MLP Classifier '''

mlp = MLPClassifier(activation='relu', random_state=123, hidden_layer_sizes=(5, 5), max_iter=500)

print(" " * 30 + "Acc " + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    avg_results_lda  = run_cross_validation(no_folds, mlp, x_lda, targets_lda[Y])
    print("LDA + MLP: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])
    end = int(round(time.time() * 1000))

print("ms " + str(end - start))


                              Acc  | Prec  |  Rec  |  F1
LDA + MLP: results for Y1  | 0.934 | 0.929 | 0.929 | 0.929
LDA + MLP: results for Y2  | 0.938 | 0.943 | 0.930 | 0.936
LDA + MLP: results for Y3  | 0.916 | 0.905 | 0.922 | 0.913
ms 3211


In [32]:
''' Cross-validation after LDA feature extraction: MLP Classifier and Label Powerset '''

start = int(round(time.time() * 1000))
mlp_powerset = LabelPowerset(MLPClassifier(activation='relu', random_state=123, hidden_layer_sizes=(5, 5), max_iter=500))
print(run_crossvalidation_with_multilabel(no_folds, mlp_powerset, x_lda, Y_lda_all))
end = int(round(time.time() * 1000))

print("ms " + str(end - start))


('0.807', '0.921', '0.925', '0.922')
ms 1574


In [34]:
''' SECOND TEST CYCLE '''
''' Only LDA and no Label Powerset '''

''' LDA + SVM '''

svm_mod = svm.SVC(kernel='rbf', random_state=123)
acc, prec, rec, f1 = 0, 0, 0, 0

print(" " * 30 + "Acc " + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    
    avg_results_lda  = run_cross_validation(no_folds, svm_mod, x_lda, targets_lda[Y])
    print("LDA + SVM: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])

    # sum the metrics that will be averaged
    acc += float(avg_results_lda[0])
    prec += float(avg_results_lda[1])
    rec += float(avg_results_lda[2])
    f1 += float(avg_results_lda[3])
    
    end = int(round(time.time() * 1000))

# print the average metrics and the proc time
print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))
print("ms " + str(end - start))


                              Acc  | Prec  |  Rec  |  F1
LDA + SVM: results for Y1  | 0.934 | 0.936 | 0.925 | 0.930
LDA + SVM: results for Y2  | 0.969 | 0.967 | 0.970 | 0.969
LDA + SVM: results for Y3  | 0.915 | 0.901 | 0.923 | 0.911
0.939 0.935 0.939 0.937
ms 163


In [36]:
''' LDA + SGD '''
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(loss="hinge", penalty="l2", max_iter=5, random_state=123)
acc, prec, rec, f1 = 0, 0, 0, 0

print(" " * 30 + "Acc " + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    
    avg_results_lda  = run_cross_validation(no_folds, sgd, x_lda, targets_lda[Y])
    print("LDA + SGD: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])

    # sum the metrics that will be averaged
    acc += float(avg_results_lda[0])
    prec += float(avg_results_lda[1])
    rec += float(avg_results_lda[2])
    f1 += float(avg_results_lda[3])
    
    end = int(round(time.time() * 1000))

print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))
print("ms " + str(end - start))


                              Acc  | Prec  |  Rec  |  F1
LDA + SGD: results for Y1  | 0.890 | 0.909 | 0.854 | 0.879
LDA + SGD: results for Y2  | 0.900 | 0.900 | 0.897 | 0.897
LDA + SGD: results for Y3  | 0.887 | 0.871 | 0.908 | 0.886
0.892 0.893 0.886 0.887
ms 31


In [37]:
''' LDA + KNN '''

knn = KNeighborsClassifier(n_neighbors=5) # KNeighborsClassifier has no random_state
acc, prec, rec, f1 = 0, 0, 0, 0

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    
    avg_results_lda  = run_cross_validation(no_folds, knn, x_lda, targets_lda[Y])
    print("LDA + KN: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])

    # sum the metrics that will be averaged
    acc += float(avg_results_lda[0])
    prec += float(avg_results_lda[1])
    rec += float(avg_results_lda[2])
    f1 += float(avg_results_lda[3])
    
    end = int(round(time.time() * 1000))

print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))
print("ms " + str(end - start))


                              Acc | Prec  |  Rec  |  F1
LDA + KN: results for Y1  | 0.934 | 0.934 | 0.925 | 0.929
LDA + KN: results for Y2  | 0.963 | 0.960 | 0.966 | 0.963
LDA + KN: results for Y3  | 0.896 | 0.888 | 0.895 | 0.890
0.931 0.927 0.929 0.927
ms 101


In [40]:
''' LDA + NB '''
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
acc, prec, rec, f1 = 0, 0, 0, 0

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    
    avg_results_lda  = run_cross_validation(no_folds, nb, x_lda, targets_lda[Y])
    print("LDA + NB: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])
    
    # sum the metrics that will be averaged
    acc += float(avg_results_lda[0])
    prec += float(avg_results_lda[1])
    rec += float(avg_results_lda[2])
    f1 += float(avg_results_lda[3])
    
    end = int(round(time.time() * 1000))
    
print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))
print("ms " + str(end - start))


                              Acc | Prec  |  Rec  |  F1
LDA + NB: results for Y1  | 0.914 | 0.911 | 0.906 | 0.908
LDA + NB: results for Y2  | 0.907 | 0.896 | 0.917 | 0.906
LDA + NB: results for Y3  | 0.902 | 0.892 | 0.904 | 0.897
0.908 0.900 0.909 0.904
ms 38


In [41]:
''' LDA + DT '''
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=123)
acc, prec, rec, f1 = 0, 0, 0, 0

print(" " * 30 + "Acc" + " | " + "Prec " + " | " + " Rec " + " | " + " F1")
for Y in range(len(targets_lda)):
    start = int(round(time.time() * 1000))
    
    avg_results_lda  = run_cross_validation(no_folds, dt, x_lda, targets_lda[Y])
    print("LDA + DT: results for Y%s " %str(Y + 1) + " | " + avg_results_lda[0] + " | " + avg_results_lda[1] + " | " + avg_results_lda[2] + " | " + avg_results_lda[3])
    
    # sum the metrics that will be averaged
    acc += float(avg_results_lda[0])
    prec += float(avg_results_lda[1])
    rec += float(avg_results_lda[2])
    f1 += float(avg_results_lda[3])
    
    end = int(round(time.time() * 1000))

print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))    
print("ms " + str(end - start))


                              Acc | Prec  |  Rec  |  F1
LDA + DT: results for Y1  | 0.916 | 0.905 | 0.918 | 0.911
LDA + DT: results for Y2  | 0.949 | 0.951 | 0.945 | 0.948
LDA + DT: results for Y3  | 0.867 | 0.865 | 0.851 | 0.857
0.911 0.907 0.905 0.905
ms 47


In [42]:
''' THIRD TEST CYCLE
    Ensemble - Cross validation + VotingClassifier '''

#estimators1 =[("randf", randf), ("knn", knn), ("dt", dt)]
#estimators2 =[("nb", nb), ("knn", knn), ("dt", dt)]
no_folds = 5
estimators = combinations([("logreg", logreg),
                           ("randf", randf),("mlp", mlp),
                           ("svm_mod", svm_mod),
                           ("sgd", sgd),
                           ("knn", knn),
                           ("nb", nb),
                           ("dt", dt)], 3)

#ensemble = VotingClassifier(estimators1, voting="hard")

for i in estimators:
    
    ensemble = VotingClassifier(list(i), voting="hard")
    print(i)
    
    avg_results1 = run_cross_validation(no_folds, ensemble, x_lda , Y1_lda)
    avg_results2 = run_cross_validation(no_folds, ensemble, x_lda , Y2_lda)
    avg_results3 = run_cross_validation(no_folds, ensemble, x_lda , Y3_lda)
    
    acc = float(avg_results1[0]) + float(avg_results2[0]) + float(avg_results3[0])
    prec = float(avg_results1[1]) + float(avg_results2[1]) + float(avg_results3[1])
    rec = float(avg_results1[2]) + float(avg_results2[2]) + float(avg_results3[2])
    f1 = float(avg_results1[3]) + float(avg_results2[3]) + float(avg_results3[3])
    
    print("{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))
    print()


(('logreg', LogisticRegression(random_state=123, solver='saga')), ('randf', RandomForestClassifier(n_estimators=50, random_state=123)), ('mlp', MLPClassifier(hidden_layer_sizes=(5, 5), max_iter=500, random_state=123)))
0.931 0.928 0.928 0.928

(('logreg', LogisticRegression(random_state=123, solver='saga')), ('randf', RandomForestClassifier(n_estimators=50, random_state=123)), ('svm_mod', SVC(random_state=123)))
0.939 0.934 0.939 0.936

(('logreg', LogisticRegression(random_state=123, solver='saga')), ('randf', RandomForestClassifier(n_estimators=50, random_state=123)), ('sgd', SGDClassifier(max_iter=5, random_state=123)))
0.922 0.918 0.920 0.918

(('logreg', LogisticRegression(random_state=123, solver='saga')), ('randf', RandomForestClassifier(n_estimators=50, random_state=123)), ('knn', KNeighborsClassifier()))
0.939 0.936 0.936 0.935

(('logreg', LogisticRegression(random_state=123, solver='saga')), ('randf', RandomForestClassifier(n_estimators=50, random_state=123)), ('nb', Gaussia

0.929 0.928 0.924 0.926

(('mlp', MLPClassifier(hidden_layer_sizes=(5, 5), max_iter=500, random_state=123)), ('knn', KNeighborsClassifier()), ('dt', DecisionTreeClassifier(random_state=123)))
0.937 0.935 0.934 0.934

(('mlp', MLPClassifier(hidden_layer_sizes=(5, 5), max_iter=500, random_state=123)), ('nb', GaussianNB()), ('dt', DecisionTreeClassifier(random_state=123)))
0.929 0.927 0.926 0.926

(('svm_mod', SVC(random_state=123)), ('sgd', SGDClassifier(max_iter=5, random_state=123)), ('knn', KNeighborsClassifier()))
0.939 0.934 0.939 0.936

(('svm_mod', SVC(random_state=123)), ('sgd', SGDClassifier(max_iter=5, random_state=123)), ('nb', GaussianNB()))
0.919 0.915 0.918 0.915

(('svm_mod', SVC(random_state=123)), ('sgd', SGDClassifier(max_iter=5, random_state=123)), ('dt', DecisionTreeClassifier(random_state=123)))
0.938 0.934 0.937 0.935

(('svm_mod', SVC(random_state=123)), ('knn', KNeighborsClassifier()), ('nb', GaussianNB()))
0.937 0.932 0.937 0.934

(('svm_mod', SVC(random_state=12

In [46]:
''' Ensemble - Cross validation + GradientBoostingClassifier '''

no_folds = 5

ensemble = GradientBoostingClassifier(loss='deviance', n_estimators=100, learning_rate=0.1, 
                                      max_depth=1, random_state=123).fit(x_lda, Y1_lda)
avg_results1 = run_cross_validation(no_folds, ensemble, x_lda , Y1_lda)
acc, prec, rec, f1 = float(avg_results1[0]), float(avg_results1[1]), float(avg_results1[2]), float(avg_results1[3])
print("Y1: " + "{:.3f}".format(acc), "{:.3f}".format(prec), "{:.3f}".format(rec), "{:.3f}".format(f1))
print()

ensemble = GradientBoostingClassifier(loss='deviance', n_estimators=100, learning_rate=0.1, 
                                      max_depth=1, random_state=123).fit(x_lda, Y2_lda)
avg_results2 = run_cross_validation(no_folds, ensemble, x_lda , Y2_lda)
acc, prec, rec, f1 = float(avg_results2[0]), float(avg_results2[1]), float(avg_results2[2]), float(avg_results2[3])
print("Y2: " + "{:.3f}".format(acc), "{:.3f}".format(prec), "{:.3f}".format(rec), "{:.3f}".format(f1))
print()

ensemble = GradientBoostingClassifier(loss='deviance', n_estimators=100, learning_rate=0.1, 
                                      max_depth=1, random_state=123).fit(x_lda, Y3_lda)
avg_results3 = run_cross_validation(no_folds, ensemble, x_lda , Y3_lda)
acc, prec, rec, f1 = float(avg_results3[0]), float(avg_results3[1]), float(avg_results3[2]), float(avg_results3[3])
print("Y3: " + "{:.3f}".format(acc), "{:.3f}".format(prec), "{:.3f}".format(rec), "{:.3f}".format(f1))
print()

acc = float(avg_results1[0]) + float(avg_results2[0]) + float(avg_results3[0])
prec = float(avg_results1[1]) + float(avg_results2[1]) + float(avg_results3[1])
rec = float(avg_results1[2]) + float(avg_results2[2]) + float(avg_results3[2])
f1 = float(avg_results1[3]) + float(avg_results2[3]) + float(avg_results3[3])
    
print("Avg: " + "{:.3f}".format(acc / 3), "{:.3f}".format(prec / 3), "{:.3f}".format(rec / 3), "{:.3f}".format(f1 / 3))


Y1: 0.941 0.941 0.936 0.938

Y2: 0.961 0.954 0.967 0.960

Y3: 0.914 0.901 0.919 0.910

Avg: 0.939 0.932 0.941 0.936
